In [ ]:
import numpy as np
from numpy import sqrt
import pandas as pd
import matplotlib.pyplot as plt

import numbers

In [ ]:
def qadd(a, b):
    return sqrt(a**2 + b**2)

def format(float):
    return f"{float:.3g}"

np.set_printoptions(formatter={'float':format})

class Measurement:
    def __init__(self, value: float | list[float] | np.ndarray, name: str | None = ""):
        value = value.astype(float) if isinstance(value, np.ndarray) else np.array(value, dtype=float)
        if value.ndim == 0:
            self.value = value.reshape((1,))
        elif value.ndim == 1:
            self.value = value
        else:
            raise ValueError(f"Measurements must be of shape (n,). Got {value.shape}.")
        self.name = name
        self.error = None

    def __repr__(self):
        if self.error is not None:
            return (
                f"Measurement({self.name}, value={self.value}, error={self.error})"
            )
        else:
            return f"Measurement({self.name}, {self.value.tolist()})"

    def setError(self, error: float | list[float] | np.ndarray):
        error = error if isinstance(error, np.ndarray) else np.array(error)
        if error.shape == tuple():
            self.error = np.full_like(self.value, error)
        elif error.shape == self.value.shape:
            self.error = error
        else:
            raise ValueError(
                f"Error must be scalar or of same shape as value. Expected () or {self.value.shape}, got {error.shape}."
            )
        return self

    def setName(self, name: str):
        self.name = name
        return self

    def group(self):
        self.name = f"({self.name})"
        return self

    def mean(self):
        if self.error is None:
            out_error = self.sem()
        else:
            syst_error = 1 / sqrt(
                np.sum(1 / self.error**2)
            )  # bei konstantem Fehler σ => = σ/\sqrt N
            out_error = sqrt(self.sem() ** 2 + syst_error**2)
        return Measurement([self.value.mean()], name=f"μ_{self.name}").setError(out_error)

    def std(self):
        # Da der Mittelwert aus den N Werten folgt, sind nur N - 1 Differenzen unabhängig. => ddof=1
        return self.value.std(ddof=1)

    def sem(self):
        return self.std() / sqrt(len(self.value))

    def __add__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out_value = self.value + other.value
        out_name = f"{self.name} + {other.name}"
        out = Measurement(out_value, out_name)

        if self.error is not None and other.error is not None:
            out.setError(qadd(self.error, other.error))
        elif self.error is not None:
            out.setError(self.error)
        elif other.error is not None:
            out.setError(other.error)

        return out


    def __mul__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))
    
        out_value = self.value * other.value
        out_name = f"{self.name} * {other.name}"
        out = Measurement(out_value, out_name)

        if self.error is not None and other.error is not None:
            out.setError(abs(out_value) * qadd(self.error / self.value, other.error / other.value))
        elif self.error is not None:
            out.setError(self.error * abs(other.value))
        elif other.error is not None:
            out.setError(other.error * abs(self.value))

        return out

    def __radd__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self + other
        out.name = f"{other.name} + {self.name}"
        return out

    def __rmul__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self * other
        out.name = f"{other.name} * {self.name}"
        return out

    def __neg__(self):
        out = self * (-1)
        out.name = f"- {self.name}"
        return out

    def __sub__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self + (-other)
        out.name = f"{self.name} - {other.name}"
        return out

    def __rsub__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))
            
        out = other + (-self)
        out.name = f"{other.name}"

In [ ]:
m = Measurement([1, 0.9234923], name="m").setError(1)
x = Measurement([2, 2], name="x")

## Bug

In [ ]:
X = Measurement([2.0, 2.0, 1.9, 2.4], name="X").setError(0.5) 
Y = Measurement([4.5, 2.9, 4.1, 3.8], name="Y").setError(0.3)

print(X + X) # Error should be 0 here but is 2**0.5 * 0.5
print(2**0.5 * 0.5)